In [0]:
# ============================================================================
# US-1.04: Global Configuration & Single Source of Truth
# ============================================================================

# ----------------------------------------------------------------------------
# 1. Base Project Naming
#    ONE variable to change the project catalog everywhere
# ----------------------------------------------------------------------------
catalog_name = "bluepeak"


# ----------------------------------------------------------------------------
# 2. Medallion and Operational Schema Names
# ----------------------------------------------------------------------------
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold   = "gold"
schema_ops    = "ops"


# ----------------------------------------------------------------------------
# 3. Unity Catalog Volume Root Path
# ----------------------------------------------------------------------------
volume_root_path = f"/Volumes/{catalog_name}/{schema_bronze}/landing_volume"


# ----------------------------------------------------------------------------
# 4. Landing Subfolder Paths
# ----------------------------------------------------------------------------
path_customers    = f"{volume_root_path}/customers"
path_accounts     = f"{volume_root_path}/accounts"
path_branches     = f"{volume_root_path}/branches"
path_transactions = f"{volume_root_path}/transactions"
path_lookup_codes = f"{volume_root_path}/lookup_codes"

# 5. Fully Qualified Bronze and Audit Table Names (Three-Level Namespaces)
table_bronze_customers    = f"{catalog_name}.{schema_bronze}.customers_raw"
table_bronze_accounts     = f"{catalog_name}.{schema_bronze}.accounts_raw"
table_bronze_branches     = f"{catalog_name}.{schema_bronze}.branches_raw"
table_bronze_transactions = f"{catalog_name}.{schema_bronze}.transactions_raw"

# Updated to use the requested ingestion_audit name
table_audit_reconcile     = f"{catalog_name}.{schema_ops}.ingestion_audit"

# ----------------------------------------------------------------------------
# 6. Session Context
#    Attempt to set catalog/schema.
#    Warn rather than fail if they don't exist yet.
# ----------------------------------------------------------------------------
session_status = (
    f"⚪ Catalog/schema not available yet. "
    f"Configured context: `{catalog_name}`.`{schema_bronze}`"
)

try:
    spark.sql(f"USE CATALOG `{catalog_name}`")
    spark.sql(f"USE SCHEMA `{schema_bronze}`")

    session_status = (
        f"🟢 Connected: Session context set to "
        f"`{catalog_name}`.`{schema_bronze}`"
    )

except Exception as error_context:
    warning_message = str(error_context).split("\n")[0]
    session_status = (
        f"⚠️ Session context not set yet: {warning_message}"
    )


# ----------------------------------------------------------------------------
# 7. Configuration Summary
# ----------------------------------------------------------------------------
print("=" * 70)
print("📐 PROJECT CONFIGURATION SUMMARY")
print("=" * 70)

print(f"🔹 Catalog Name  : {catalog_name}")
print(
    f"🔹 Schemas       : "
    f"{schema_bronze}, {schema_silver}, {schema_gold}, {schema_ops}"
)

print(f"🔹 Volume Root   : {volume_root_path}")

print("-" * 70)
print("📂 Landing Folders:")
print(f"  └─ Customers    : {path_customers}")
print(f"  └─ Accounts     : {path_accounts}")
print(f"  └─ Branches     : {path_branches}")
print(f"  └─ Transactions : {path_transactions}")
print(f"  └─ Lookup Codes : {path_lookup_codes}")

print("-" * 70)
print("🗄️ Fully Qualified Tables:")
print(f"  └─ Customers    : {table_bronze_customers}")
print(f"  └─ Accounts     : {table_bronze_accounts}")
print(f"  └─ Branches     : {table_bronze_branches}")
print(f"  └─ Transactions : {table_bronze_transactions}")
print(f"  └─ Audit Logs   : {table_audit_reconcile}")

print("-" * 70)
print(f"⚙️ Session Status : {session_status}")

print("=" * 70)

def ingest_bronze_source(target_table, source_folder_path, file_pattern="*.csv"):
    """
    Ingests raw CSV data using COPY INTO with explicit inline metadata expression mapping
    to guarantee non-null values for _source_file, _ingested_at, and _batch_id.
    """
    print(f"\n🚀 Initiating ingestion for target table: {target_table}")
    print(f"📁 Scanning source directory : {source_folder_path}")
    
    try:
        # We wrap the source inside a SELECT statement that explicitly appends 
        # file lineage and run metadata directly to every incoming row
        copy_query = f"""
        COPY INTO {target_table}
        FROM (
          SELECT 
            *,
            _metadata.file_path AS _source_file,
            current_timestamp() AS _ingested_at,
            '{current_batch_id}' AS _batch_id
          FROM '{source_folder_path}'
        )
        FILEFORMAT = CSV
        PATTERN = '{file_pattern}'
        FORMAT_OPTIONS (
          'header' = 'true',
          'inferSchema' = 'false'
        )
        COPY_OPTIONS (
          'force' = 'false',
          'mergeSchema' = 'true'
        );
        """
        
        # Execute the query and capture execution output metrics dataframe
        copy_result_df = spark.sql(copy_query)
        
        # Extract performance metrics
        metrics = copy_result_df.select("num_inserted_rows", "num_affected_rows").collect()[0]
        rows_loaded  = int(metrics["num_inserted_rows"])
        files_loaded = 1 if rows_loaded > 0 else 0
        
        print(f"✅ Ingestion Complete. Files loaded: {files_loaded} | Rows committed: {rows_loaded}")
        print(f"Pillared auditing metadata (_source_file, _ingested_at, _batch_id) injected inline.")
        
        # Log a structured summary row into your ops.ingestion_audit ledger
        audit_row = [(
            current_batch_id,
            target_table,
            source_folder_path,
            files_loaded,
            rows_loaded
        )]
        
        audit_df = spark.createDataFrame(
            audit_row, 
            schema="batch_id STRING, target_table STRING, source_path STRING, files_loaded LONG, rows_loaded LONG"
        ).withColumn("load_ts", current_timestamp())
        
        audit_df.write.format("delta").mode("append").saveAsTable(table_audit_reconcile)
        print(f"⚙️ Operational audit log successfully recorded in database.")
        
    except Exception as execution_error:
        print(f"❌ FATAL ERROR DURING INGESTION: {str(execution_error)}")
        raise execution_error
